# interlingua — walkthrough

Reproducible end-to-end pass through the pipeline that produces the conlang lexicon.

**This notebook is read-mostly by default.** All cells assume the heavy pipeline stages have already been run and their artifacts are on disk under `/media/menser/fauna/interlingua/data/`. To regenerate the artifacts from scratch:

```bash
source .venv/bin/activate
python -m conlang.slice --sae-release gemma-scope-2b-pt-res-canonical \
    --sae-id layer_12/width_16k/canonical \
    --neuronpedia-model gemma-2-2b \
    --neuronpedia-source 12-gemmascope-res-16k \
    --top-n 1000 --dedup-method hdbscan
python -m conlang.run_coactivation --use-flores --n-per-lang 1000
python -m conlang.regularize
python -m conlang.lexicon
python -m conlang.site
```

The coactivation stage needs a GPU and takes a few minutes; the others finish in seconds.

In [1]:
import json
import numpy as np

from conlang import INTERIM_DIR, PROCESSED_DIR, RAW_DIR
from conlang.phonology import (
    apply_class_prefix, negate, CLASS_PREFIXES,
    VOWELS, SINGLE_CONSONANTS,
)


## Stage 1–2 — Ingest and dedupe

1000 SAE features pass the §6 filter from the bulk Neuronpedia explanation dump. We compute pairwise cosine similarity over their decoder vectors and cluster with HDBSCAN.

In [2]:
features = [json.loads(line) for line in open(RAW_DIR / 'features.jsonl')]
sim = np.load(INTERIM_DIR / 'sim_matrix.npy')
labels = np.load(INTERIM_DIR / 'hdbscan_labels.npy')
print(f'{len(features)} features')
print(f'sim matrix: {sim.shape}')
print(f'HDBSCAN: {len(set(labels.tolist()))} groups, '
      f'{int((labels == -1).sum())} noise singletons')
print()
print('first three feature labels:')
for f in features[:3]:
    print(f'  [{f["feature_id"]:>5d}] {f["label"]}')


1000 features
sim matrix: (1000, 1000)
HDBSCAN: 8 groups, 838 noise singletons

first three feature labels:
  [    1] instructions related to cooking and food preparation
  [    2] terms related to health and healthy living
  [    3] phrases related to optimization and efficiency


## Stage 3 — Co-activation edges

5982 FLORES-200 dev sentences across six languages (eng, fra, deu, spa, zho, jpn) pass through Gemma 2 2B with the SAE encoder hooked at layer 12. We accumulate pairwise feature co-firing and normalize to PMI.

In [3]:
pmi = np.load(INTERIM_DIR / 'coactivation' / 'pmi.npy')
top_pairs = json.load(open(INTERIM_DIR / 'coactivation' / 'top_pairs.json'))
print(f'PMI matrix: {pmi.shape}, {int((pmi > 0).sum())} positive pairs')
print(f'top-50 pairs stashed in top_pairs.json')
print()
print('top 5 PMI pairs:')
for pair in top_pairs[:5]:
    print(f'  PMI {pair["pmi"]:+.2f}, cofire={pair["cofire_count"]}')
    print(f'    {pair["label_a"][:60]!r}')
    print(f'    {pair["label_b"][:60]!r}')


PMI matrix: (1000, 1000), 221618 positive pairs
top-50 pairs stashed in top_pairs.json

top 5 PMI pairs:
  PMI +7.65, cofire=3
    'terms and symbols commonly used in mathematical expressions '
    'mathematical expressions and symbols related to vectors and '
  PMI +7.03, cofire=3
    'references to individuals and their life events'
    'biographical details related to birth and early life'
  PMI +6.83, cofire=15
    'mathematical expressions or formulas'
    'mathematical expressions and equations'
  PMI +6.57, cofire=3
    'mathematical symbols and constructs used in formal proofs an'
    'references to location and burial contexts'
  PMI +6.46, cofire=6
    'quoted speech or dialogue in the text'
    'dialogues and emotional expressions in text'


## Stage 4 — Decision gate

Three load-bearing questions before continuing to morphology.

In [4]:
n_distinct = len(features) - int((labels >= 0).sum() - len(set(l for l in labels if l >= 0)))
n_with_pmi_parent = int((pmi.max(axis=1) > 0).sum())
crystal_coverage = 0.0  # See spec §7: bridge failed, 0% margin >= 0.05

print(f'Distinct semantic fields:       {n_distinct} (target 500-5000) — OK')
print(f'Crystal coverage at margin 0.05: {crystal_coverage:.0%} (target >= 30%) — FAIL')
print(f'Nodes with positive-PMI parent: {n_with_pmi_parent}/{len(features)} — OK')
print()
print('2/3 green. The crystal failure is exactly the Commitment 7')
print('failure-mode the spec anticipated; mitigation (compositional')
print('negation handled by morphology) is in force.')


Distinct semantic fields:       845 (target 500-5000) — OK
Crystal coverage at margin 0.05: 0% (target >= 30%) — FAIL
Nodes with positive-PMI parent: 981/1000 — OK

2/3 green. The crystal failure is exactly the Commitment 7
failure-mode the spec anticipated; mitigation (compositional
negation handled by morphology) is in force.


## Stage 5 — Regularize

Collapse the Stage 3 multigraph into per-node `parent` (highest-PMI neighbor), `siblings` (HDBSCAN cluster co-members), and `near` (top cosine neighbors).

In [5]:
reg = json.load(open(PROCESSED_DIR / 'regularized.json'))
nodes = reg['nodes']
n_with_parent = sum(1 for n in nodes if n['parent'])
n_with_siblings = sum(1 for n in nodes if n['siblings'])
print(f'{n_with_parent}/{len(nodes)} nodes have a parent')
print(f'{n_with_siblings}/{len(nodes)} nodes have at least one sibling')
print()
print('example node:')
import textwrap
print(textwrap.indent(json.dumps(nodes[491], indent=2)[:600], '  '))


981/1000 nodes have a parent
162/1000 nodes have at least one sibling

example node:
  {
    "slice_idx": 491,
    "feature_id": 677,
    "label": "words indicating roles, responsibilities, or actions related to managing and improving practices",
    "parent": {
      "slice_idx": 48,
      "pmi": 3.313706636428833
    },
    "siblings": [
      28,
      33,
      81,
      112,
      171,
      173,
      184,
      187,
      223,
      230,
      289,
      293,
      306,
      310,
      340,
      355,
      379,
      392,
      402,
      418,
      432,
      454,
      477,
      487,
      529,
      651,
      702,
      821,
      833,
      865,
      870,
      951,
      969,
      976,
      992
    ],
    "near": [
      {
        "slice_idx": 165,
   


## Stage 6 — Phonology, lexicon, site

Eleven Bantu noun classes, productive `si-` negation, phonosemantic stems (CV1=cluster, CV2=parent, CV3=self).

In [6]:
# Show the 11-class affix paradigm on a sample stem.
for cid in sorted(CLASS_PREFIXES):
    pfx, desc = CLASS_PREFIXES[cid]
    surface = apply_class_prefix('paka', cid)
    ant = negate(surface)
    print(f'class {cid:2d} ({pfx:>3}-) {desc[:30]:<30} → {surface:<10} / {ant}')


class  1 ( mu-) human, singular                → mupaka     / simupaka
class  2 ( ba-) human, plural                  → bapaka     / sibapaka
class  3 (  u-) plant/object, singular         → upaka      / supaka
class  4 ( mi-) plant/object, plural           → mipaka     / simipaka
class  5 ( li-) fruit/paired thing, singular   → lipaka     / silipaka
class  6 ( ma-) fruit/paired thing, plural     → mapaka     / simapaka
class  7 ( ki-) tool/thing, singular           → kipaka     / sikipaka
class  8 ( vi-) tool/thing, plural             → vipaka     / sivipaka
class  9 (  n-) animal/language, singular (hom → mpaka      / simpaka
class 10 ( zi-) animal/language, plural        → zipaka     / sizipaka
class 11 ( lu-) long thing / mass / abstract   → lupaka     / silupaka


In [7]:
lex = json.load(open(PROCESSED_DIR / 'lexicon.json'))
entries = lex['entries']
by_class = {}
for e in entries:
    by_class[e['class_id']] = by_class.get(e['class_id'], 0) + 1
print(f'{len(entries)} lexicon entries\n')
print('class distribution:')
for cid in sorted(by_class):
    print(f'  class {cid:2d} ({CLASS_PREFIXES[cid][1]:<35}): {by_class[cid]}')


1000 lexicon entries

class distribution:
  class  1 (human, singular                    ): 61
  class  5 (fruit/paired thing, singular       ): 8
  class  7 (tool/thing, singular               ): 220
  class  9 (animal/language, singular (homorganic stop or yi-)): 6
  class 11 (long thing / mass / abstract       ): 705


In [8]:
import random
random.seed(7)
print('sample lexicon entries (random):\n')
for e in random.sample(entries, 8):
    print(f'  {e["surface"]:<14} (neg {e["antonym"]:<14}) '
          f'class={e["class_id"]:2d}  {e["label"][:55]!r}')


sample lexicon entries (random):

  muwaloma       (neg simuwaloma    ) class= 1  'symbols and characters related to mathematical expressi'
  kinevavi       (neg sikinevavi    ) class= 7  'references to different contexts in a structured format'
  luwamoye       (neg siluwamoye    ) class=11  'technical terms and descriptors in scientific or engine'
  luwanubu       (neg siluwanubu    ) class=11  'gerunds and present participles in various contexts'
  luwadipuya     (neg siluwadipuya  ) class=11  'details related to professional experience and qualific'
  kiwadipa       (neg sikiwadipa    ) class= 7  'tokens and symbols related to mathematical or statistic'
  luwadideyi     (neg siluwadideyi  ) class=11  'phrases indicating excessiveness or extreme qualities'
  lubotani       (neg silubotani    ) class=11  'references to geographic locations'


## Open the rendered deliverables

After running `python -m conlang.site && mkdocs build`:

- **Single-page lexicon** (Phase A): `docs/static/lexicon.html` (also served at `site/static/lexicon.html` from the MkDocs build).
- **Multi-page site** (Phase B): `data/processed/site/index.html`.